In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    
from config import *
print(f"Environment linked. Data dir: {DATA_DIR}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

print(f"Loading dataset from: {MACHINE_A_DATASET_FILE}")
df = pd.read_csv(MACHINE_A_DATASET_FILE)

features_to_plot = MACHINE_A_FEATURES
print(f"Visualizing Features: {features_to_plot}")

sns.set_theme(style="whitegrid")

In [ ]:
plt.figure(figsize=(6, 4))

# FIX: Add hue='label' and legend=False
ax = sns.countplot(x='label', data=df, hue='label', palette='viridis', legend=False)

# Add counts on top of bars
for p in ax.patches:
    height = p.get_height()
    # Safety check for NaN heights (rare plotting artifact)
    if height > 0:
        ax.annotate(f'{int(height)}', (p.get_x() + 0.35, height + 100))

plt.title("Class Distribution (0=Clean, 1=Jamming)")
plt.xlabel("Class Label")
plt.ylabel("Count")
plt.show()

In [ ]:
# Create a grid of plots based on how many features we have
num_features = len(features_to_plot)
cols = 2
rows = (num_features + 1) // cols

plt.figure(figsize=(15, 5 * rows))

for i, feature in enumerate(features_to_plot):
    plt.subplot(rows, cols, i + 1)
    
    # Plot Clean (Blue) vs Jamming (Orange)
    sns.kdeplot(data=df, x=feature, hue='label', fill=True, 
                palette={0: 'tab:blue', 1: 'tab:orange'}, common_norm=False)
    
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    
plt.tight_layout()
plt.show()

In [ ]:
# Sample data for speed
plot_sample = df.sample(n=min(2000, len(df)), random_state=42)

# Create Pair Plot
g = sns.pairplot(
    plot_sample, 
    vars=features_to_plot, 
    hue='label', 
    palette={0: 'tab:blue', 1: 'tab:orange'},
    plot_kws={'alpha': 0.6, 's': 15},
    diag_kind='kde'
)

g.fig.suptitle("Feature Interaction Matrix", y=1.02)
plt.show()

In [ ]:
# Check if we have the metadata columns
if 'jamming_power' in df.columns:
    
    # Filter for Jamming samples only (Clean has power=0)
    jamming_df = df[df['label'] == 1].copy()
    
    # We only care about known powers (exclude -1)
    jamming_df = jamming_df[jamming_df['jamming_power'] >= 0]

    if len(jamming_df) > 0:
        plt.figure(figsize=(12, 5))
        
        # Plot: Calculated Power vs. Actual Jamming Power Label
        y_feature = 'Mean Power' if 'Mean Power' in features_to_plot else features_to_plot[0]
        
        # FIX: Add hue='jamming_power' and legend=False
        sns.boxplot(
            x='jamming_power', 
            y=y_feature, 
            data=jamming_df, 
            hue='jamming_power', 
            palette='Reds', 
            legend=False
        )
        
        plt.title(f"Sensitivity Check: {y_feature} vs. Metadata Jamming Power")
        plt.xlabel("True Jamming Power (from Metadata)")
        plt.ylabel(f"Calculated {y_feature}")
        plt.show()
        
    else:
        print("No valid jamming power metadata found.")
else:
    print("Metadata column 'jamming_power' not found in dataset.")

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[features_to_plot].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
def calc_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                       for i in range(len(X.columns))]
    return vif_data.sort_values(by="VIF", ascending=False)

print(calc_vif(df[features_to_plot]))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION SECTION ---
# The current "Winning" list
BASE_FEATURES = [
    'Mean Power', 
    'PAPR', 
    'Kurtosis', 
    'Skewness', 
    'Spectral Flatness'
]

# Which features do you want to try deleting? (The ones with high VIF)
# We test removing them one by one.
SUSPECTS_TO_DROP = ['PAPR', 'Mean Power'] 

# --- THE DYNAMIC ENGINE ---
def evaluate_feature_sets(X_df, y, base_list, drop_list):
    """
    Automatically generates scenarios:
    1. The Baseline (All features)
    2. Baseline minus Suspect A
    3. Baseline minus Suspect B
    ...
    """
    
    # Create the dictionary of scenarios dynamically
    scenarios = {"Baseline (All 5)": base_list}
    
    for suspect in drop_list:
        if suspect in base_list:
            # Create a list without this specific suspect
            subset = [f for f in base_list if f != suspect]
            scenarios[f"Remove {suspect}"] = subset
            
    print(f"{'SCENARIO':<25} | {'F1 SCORE':<10} | {'STD DEV':<10} | {'NUM FEATS'}")
    print("-" * 65)

    for name, features in scenarios.items():
        # Select data
        X_subset = X_df[features]
        
        # PIPELINE: Scale -> Train
        # Scaling is crucial when comparing Power (high val) vs Flatness (low val)
        model = make_pipeline(
            StandardScaler(), 
            LogisticRegression(class_weight='balanced', max_iter=1000)
        )
        
        # Run CV
        scores = cross_val_score(model, X_subset, y, cv=5, scoring='f1')
        
        # Report
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        print(f"{name:<25} | {mean_score:.4f}     | +/-{std_score:.4f} | {len(features)}")

# --- EXECUTION ---
print("--- VIF High-Value Check ---")
evaluate_feature_sets(df, df['label'], features_to_plot, SUSPECTS_TO_DROP)